# 03 · Trust & Uncertainty

**Forecast vs Instinct** - Corporación Favorita (Ecuador), 12 stores × 8 families = 96 daily series, 14-day forecasts.

Notebook 02 picked a champion forecast (LightGBM, known promo plan, in every family). A point forecast alone is not
enough for a planner, who also needs to know:

1. **How wrong could it be?** → an **80 % prediction interval** (P10-P90) for every series-day.
2. **Which forecasts deserve a human look first?** → the **Forecast Trust Index**, a 0-100 score with three bands
   (Accept / Monitor / Review).

Both were fitted on the **calibration** stage (8 origins, late April - June 2017), frozen with SHA-256 hashes, and
then checked once on the **holdout** (6 origins, 28 June - 15 August 2017).

Ground rules:

- **Intervals are empirical, not guarantees.** "80 %" is a target measured on past errors; there is no promise any
  single interval contains the actual.
- **The Trust score is an index, not a probability.** A score of 85 does not mean "85 % chance the forecast is
  right"; it ranks series so review time goes where it is most needed.
- Sales are observed sales, not unconstrained demand; promo = count of promoted items (associations only).
- Nothing is re-trained or re-run: every number is computed here from saved backtests and frozen files.

In [1]:
import sys, json, hashlib, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from src import config as C, metrics as M
from src.backtest import evaluable, masked_view
from src.runner import load_backtests
from src.selection import selection_table, pick_champions, champion_rows
from src import indexing as IX  # per-series values are shown as an index (Kaggle rule 7B: no raw data)

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)

OUT = C.OUTPUTS
J = lambda name: json.loads((OUT / name).read_text())

# Same family colours as notebook 01; one fixed colour per model (never re-assigned)
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
FAM_COLOR = dict(zip(C.FAMILIES, PALETTE))
MODEL_ORDER = ["naive", "seasonal_naive", "moving_average", "holt_winters", "lightgbm A", "lightgbm B"]
MODEL_COLOR = {"naive": "#8a8984", "seasonal_naive": "#eda100", "moving_average": "#e87ba4",
               "holt_winters": "#2a78d6", "lightgbm A": "#eb6834", "lightgbm B": "#6250d6"}
BAND_COLOR = {"Accept": "#1baf7a", "Monitor": "#eda100", "Review": "#e34948"}
LAYOUT = dict(template="simple_white", height=380, margin=dict(l=50, r=20, t=50, b=40),
              font=dict(size=12), legend=dict(title=None))

bt = load_backtests()                                   # every model, every stage, read-only
bt["mv"] = np.where(bt["model"] == "lightgbm", "lightgbm " + bt["variant"], bt["model"])
panel = pd.read_parquet(C.PROCESSED / "panel.parquet")
cal = json.loads((C.CONFIG / "split_calendar.json").read_text())

def table(df, by, models=MODEL_ORDER, value="wape"):
    """Ratio-of-sums metric per group x model (src.metrics.grouped_metrics), models as columns."""
    g = M.grouped_metrics(df, list(by) + ["mv"])
    return g.pivot_table(index=list(by), columns="mv", values=value)[[m for m in models if m in set(g.mv)]]

print(f"backtest rows: {len(bt):,}  |  stages: {sorted(bt.stage.unique())}  |  models: {sorted(bt.mv.unique())}")

from scipy import stats
from src import trust as TR
from src.intervals import fit_quantiles, apply_intervals, bucket, origin_scales
sel = J("selection.json"); champs = sel["champions"]
ivj = J("intervals.json"); Q = pd.DataFrame(ivj["quantiles"])
tw = J("trust_weights.json"); tcal = J("trust_calibration.json"); res = J("holdout_results.json")
ci = pd.read_parquet(OUT / "holdout" / "champion_intervals.parquet")
ts = pd.read_parquet(OUT / "holdout" / "trust_scores.parquet")
hold_origins = pd.to_datetime(cal["stages"]["holdout"])
print("champion per family:", {f: c["model"] for f, c in champs.items()})

backtest rows: 201,600  |  stages: ['calibration', 'forward', 'holdout', 'selection']  |  models: ['holt_winters', 'lightgbm A', 'lightgbm B', 'moving_average', 'naive', 'seasonal_naive']


champion per family: {'BEVERAGES': 'lightgbm', 'BREAD/BAKERY': 'lightgbm', 'CLEANING': 'lightgbm', 'DAIRY': 'lightgbm', 'FROZEN FOODS': 'lightgbm', 'GROCERY I': 'lightgbm', 'PERSONAL CARE': 'lightgbm', 'PRODUCE': 'lightgbm'}


## 1. How the intervals are built: scaled split-conformal

The idea is simple: **look at how wrong the champion was on the calibration windows, and assume the future will be
wrong in a similar way.**

1. For every calibration row compute a **scaled residual** r = (actual − forecast) / scale_T, where scale_T is the
   series' mean sales over the 28 days up to the origin (known at T). Scaling lets one quantile serve a big store and
   a small one.
2. Group residuals by **family × horizon bucket** (days 1-7, days 8-14): 8 × 2 = 16 groups, 672 residuals each
   (8 origins × 12 stores × 7 days).
3. In each group take the **10th and 90th percentiles**, q_lo and q_hi.
4. For a new forecast F: **P10 = max(0, F + q_lo × scale_T)**, **P90 = max(0, F + q_hi × scale_T)**
   (the point forecast is kept inside its band).

The cell refits the quantiles from the saved calibration backtests and checks them against the frozen
`intervals.json`.

In [2]:
cal_rows = champion_rows(bt[bt.stage == "calibration"], champs, "A")
q_re = fit_quantiles(cal_rows, panel)
chk = q_re.merge(Q, on=["family", "bucket"], suffixes=("", "_frozen"))
print("max |refit - frozen| quantile:", float(np.abs(chk[["q_lo", "q_hi"]].values - chk[["q_lo_frozen", "q_hi_frozen"]].values).max().round(6)))
qt = Q.pivot(index="family", columns="bucket", values=["q_lo", "q_hi"]).loc[C.FAMILIES]
qt[("width", "1-7")] = qt[("q_hi", "1-7")] - qt[("q_lo", "1-7")]
qt[("width", "8-14")] = qt[("q_hi", "8-14")] - qt[("q_lo", "8-14")]
print(ivj["method"])
qt.round(3)

max |refit - frozen| quantile: 0.0
Scaled split-conformal: residual (A-F)/scale_T, scale_T = mean sales of the 28 days up to T; quantiles 0.10/0.90 per family x horizon bucket from CALIBRATION champion backtests (8 origins). P10/P90 = max(0, F + q*scale_T).


q_lo          q_hi         width       
bucket           1-7   8-14    1-7   8-14    1-7   8-14
family                                                 
BEVERAGES     -0.163 -0.175  0.227  0.221  0.391  0.396
DAIRY         -0.164 -0.178  0.172  0.149  0.336  0.327
BREAD/BAKERY  -0.133 -0.134  0.214  0.205  0.347  0.339
GROCERY I     -0.153 -0.167  0.210  0.214  0.363  0.381
FROZEN FOODS  -0.321 -0.335  0.365  0.340  0.686  0.675
CLEANING      -0.234 -0.235  0.257  0.197  0.491  0.432
PERSONAL CARE -0.230 -0.228  0.252  0.254  0.482  0.482
PRODUCE       -0.132 -0.128  0.155  0.142  0.287  0.270

**Reading it:** in most family-buckets q_hi is larger in size than q_lo, so the bands are **asymmetric**: the
champion under-forecast more often than it over-forecast on calibration (its calibration bias was about −2.4 %, see
notebook 02), and the band leans upwards to absorb that. **FROZEN FOODS** gets the widest band (about −33 % to +35 %
of a typical recent day), PRODUCE the narrowest, then DAIRY and BREAD/BAKERY. Week 2 is not systematically wider than week 1, which matches the flat horizon profile in
notebook 02.

## 2. Did the intervals hit 80 %? Calibration vs holdout

Three coverage numbers (share of actuals inside P10-P90):

- **in-sample (calibration):** the quantiles scored on the same rows they were fitted on - close to 80 % by construction;
- **leave-one-origin-out (calibration):** refit the quantiles 8 times, each time leaving one origin out and scoring it -
  an honest preview of out-of-sample coverage;
- **holdout:** the frozen quantiles applied to the 6 holdout origins, run once.

In [3]:
ins = apply_intervals(evaluable(cal_rows), panel, q_re)
ins["inside"] = (ins.actual >= ins.p10) & (ins.actual <= ins.p90)
loo = []
for o in sorted(cal_rows.origin.unique()):
    qo = fit_quantiles(cal_rows[cal_rows.origin != o], panel)
    loo.append(apply_intervals(evaluable(cal_rows[cal_rows.origin == o]), panel, qo))
loo = pd.concat(loo)
loo["inside"] = (loo.actual >= loo.p10) & (loo.actual <= loo.p90)
ho = evaluable(ci[ci.stage == "holdout"]).copy()
ho["inside"] = (ho.actual >= ho.p10) & (ho.actual <= ho.p90)
ho["below"], ho["above"] = ho.actual < ho.p10, ho.actual > ho.p90
print(f"in-sample      {ins.inside.mean()*100:.2f}%   (intervals.json: {ivj['calibration_coverage_in_sample']:.2f}%)")
print(f"leave-one-out  {loo.inside.mean()*100:.2f}%   (intervals.json: {ivj['calibration_coverage_leave_one_origin_out']:.2f}%)")
print(f"holdout        {ho.inside.mean()*100:.2f}%   (holdout_results.json: {res['champion_system']['coverage']:.2f}%)")
print(f"holdout misses: {ho.below.mean()*100:.1f}% below P10, {ho.above.mean()*100:.1f}% above P90 (target 10% / 10%)")
cov = pd.DataFrame({"calibration in-sample": ins.groupby("family").inside.mean(),
                    "calibration leave-one-origin-out": loo.groupby("family").inside.mean(),
                    "holdout": ho.groupby("family").inside.mean(),
                    "holdout below P10": ho.groupby("family").below.mean(),
                    "holdout above P90": ho.groupby("family").above.mean()}).loc[C.FAMILIES] * 100
cov.round(1)

in-sample      79.76%   (intervals.json: 79.76%)
leave-one-out  79.20%   (intervals.json: 79.20%)
holdout        80.32%   (holdout_results.json: 80.32%)
holdout misses: 10.5% below P10, 9.2% above P90 (target 10% / 10%)


,calibration in-sample,calibration leave-one-origin-out,holdout,holdout below P10,holdout above P90
family,,,,,
BEVERAGES,79.8,78.3,72.2,12.4,15.4
DAIRY,79.8,79.3,83.9,10.9,5.2
BREAD/BAKERY,79.8,80.0,75.1,14.8,10.1
GROCERY I,79.8,79.2,87.8,6.1,6.2
FROZEN FOODS,79.8,79.2,81.1,9.5,9.4
CLEANING,79.8,79.3,75.9,11.5,12.6
PERSONAL CARE,79.8,78.5,86.4,6.3,7.2
PRODUCE,79.8,79.8,80.2,12.6,7.2


In [4]:
fig = go.Figure()
for name, col, dash in [("calibration leave-one-origin-out", "#8a8984", "dot"), ("holdout", "#e34948", "solid")]:
    s = cov[name]
    fig.add_scatter(x=s.round(1), y=s.index, mode="markers", name=name, marker=dict(size=11, color=col, symbol="circle" if name == "holdout" else "diamond"))
fig.add_vline(x=80, line_dash="dash", line_color="#52514e", annotation_text="80% target", annotation_position="top")
fig.update_layout(**{**LAYOUT, "height": 380}, title="P10-P90 coverage by family", xaxis_title="coverage %",
                  yaxis=dict(autorange="reversed"), xaxis_range=[55, 100])
fig.show()

**Reading it:** overall, the holdout coverage landed very close to the 80 % target, in line with the
leave-one-origin-out preview, and the misses split almost evenly between below P10 and above P90.
**The overall number hides a family spread**:

- **under-covered:** BEVERAGES (misses on both sides, more often above P90), BREAD/BAKERY (mostly below P10) and
  CLEANING - the bands were too narrow for these families in summer;
- **over-covered:** GROCERY I, PERSONAL CARE and DAIRY - the bands were wider than needed, consistent with these
  families' lower holdout WAPE than on calibration (GROCERY I, PERSONAL CARE).

Coverage is an average property over many rows; it says nothing about a single series-day.

### Coverage by horizon

In [5]:
covh = pd.DataFrame({"calibration leave-one-origin-out": loo.groupby("horizon").inside.mean() * 100,
                     "holdout": ho.groupby("horizon").inside.mean() * 100})
fig = go.Figure()
for name, col in [("calibration leave-one-origin-out", "#8a8984"), ("holdout", "#e34948")]:
    fig.add_scatter(x=covh.index, y=covh[name].round(1), name=name, mode="lines+markers", line=dict(color=col, width=2), marker=dict(size=8))
fig.add_hline(y=80, line_dash="dash", line_color="#52514e")
fig.update_layout(**LAYOUT, title="P10-P90 coverage by horizon", xaxis=dict(title="horizon (days)", dtick=1),
                  yaxis_title="coverage %", yaxis_range=[50, 100])
fig.show()
covh.round(1).T

horizon,1,2,3,4,5,6,7,8,9,10,11,12,13,14
calibration leave-one-origin-out,85.7,84.8,82.3,77.0,59.0,81.5,86.1,84.8,83.5,78.4,74.9,61.2,84.4,85.5
holdout,87.5,88.5,83.3,73.4,70.8,80.4,83.9,85.8,88.4,81.4,70.7,67.2,81.1,82.1


**Reading it:** there is no steady widening problem with the horizon, but there is a clear **weekday pattern**: the
two dips, at horizons 5 and 12, are both **Sundays** (origins are Tuesdays), in calibration and holdout alike. Sunday
is the peak sales day (notebook 01), its errors are larger, yet the quantiles are pooled over all seven days of a
bucket, so Sunday bands are too narrow and weekday bands slightly too wide. Quantiles by weekday (or scaled by the
forecast instead of the 28-day mean) would be the natural next improvement. Each horizon point rests on 576 rows
from only 6 calendar days, so individual points are noisy.

## 3. How wide are the bands?

Width = P90 − P10 on the index scale (÷ the series' training-period mean daily sales × 100, so the width is in
"index points" of an average day), and relative to the point forecast (Σ width ÷ Σ forecast) so families can be
compared. Per-family unit levels are not shown (Kaggle rules); only the all-96-series portfolio width is in units.

In [6]:
ho["width"] = ho.p90 - ho.p10
_base = ho.series_id.map(IX.series_base(panel))
ho["width_index"], ho["forecast_index"] = IX.to_index(ho.width, _base), IX.to_index(ho.forecast, _base)
wd = ho.groupby("family").apply(lambda g: pd.Series({
    "mean width (index points)": g.width_index.mean(), "mean forecast (index)": g.forecast_index.mean(),
    "width / forecast %": 100 * g.width.sum() / g.forecast.sum(), "coverage %": 100 * g.inside.mean()})).loc[C.FAMILIES]
print(f"Champion system (all 96 series), holdout: mean width {M.mean_width(ho.p10, ho.p90):.1f} units "
      f"(holdout_results.json: {res['champion_system']['mean_width']:.1f}), {ho.width_index.mean():.1f} index points; "
      f"width / forecast {100 * ho.width.sum() / ho.forecast.sum():.1f}%")
display(wd.round(1))
tot = pd.read_parquet(OUT / "holdout" / "chain_total.parquet")
th = tot[(tot.stage == "holdout") & (tot.closed == 0) & tot.actual.notna()]
print(f"Chain total (sum of all 96 series, own quantiles): coverage {M.coverage(th.actual, th.p10, th.p90):.1f}% over {len(th)} days, "
      f"width / forecast {100 * (th.p90 - th.p10).sum() / th.forecast.sum():.1f}%")

Champion system (all 96 series), holdout: mean width 615.6 units (holdout_results.json: 615.6), 40.9 index points; width / forecast 37.3%


,mean width (index points),mean forecast (index),width / forecast %,coverage %
family,,,,
BEVERAGES,47.4,119.3,39.7,72.2
DAIRY,35.4,106.2,33.5,83.9
BREAD/BAKERY,34.1,99.8,34.4,75.1
GROCERY I,39.6,106.2,37.4,87.8
FROZEN FOODS,47.6,69.2,68.4,81.1
CLEANING,48.0,104.3,46.1,75.9
PERSONAL CARE,46.0,94.3,48.8,86.4
PRODUCE,29.4,105.1,28.1,80.2


Chain total (sum of all 96 series, own quantiles): coverage 83.3% over 84 days, width / forecast 15.9%


**Reading it:** a band spans from about 28 % of the daily forecast (PRODUCE) to about 68 % (FROZEN FOODS), 37 %
overall. The **chain total** has its own band (fitted on total-level residuals),
because adding 96 series' P10s does not give the P10 of the sum: errors partly cancel, so the total band is far
narrower in relative terms.

## 4. The Forecast Trust Index: components and formulas

The question it answers at origin T: *how much should I trust this series' 14-day forecast?* Every input is known at
the end of day T. Five components, each 0-100 (higher = more trustworthy):

| Component | Formula | What it captures |
|---|---|---|
| Historical accuracy | 100 × (1 − min(WAPE_hist, 50) / 50) | champion WAPE for this series over the last ≤ 8 backtest windows already complete at T |
| Interval reliability | 100 × (1 − min(\|cov_hist − 80\|, 40) / 40) | did the P10-P90 band hit 80 % on those windows? |
| Demand stability | 100 × (1 − min(CV_56, 1)) × (1 − zero_share_56) | how erratic daily sales were over the last 56 days |
| Model agreement | 100 × (1 − min(cv_models / 0.5, 1)) | spread (std ÷ mean) of the 5 models' 14-day totals |
| Data completeness | 100 × n_ok_56 / 56 | share of the last 56 days that are open with an observed sale |

**Score** = Σ weight × component (weights sum to 1). **Bands:** Accept ≥ 80, Monitor 60-80, Review < 60.
**Hard flags force Review** regardless of score: completeness < 80 %, a chain closure (25 Dec / 1 Jan) in the window,
or a planned promo intensity above the series' historical 95th percentile (the forecast would be extrapolating).
**Priority** for the exception inbox = (100 − score) × the series' share of the total 14-day forecast.

Full definitions: `docs/agent_notes/trust_methodology.md`; code: `src/trust.py`. Frozen settings:

In [7]:
print("Frozen weights :", tw["weights"])
print("Default weights:", C.TRUST_WEIGHTS_DEFAULT)
print("Bands          :", [(b["band"], b["min"]) for b in tw["bands"]])
print("Constants      :", {k: v for k, v in tw["constants"].items() if k in ("wape_cap", "cov_target", "cov_tol", "cv_cap", "agree_cap", "lookback_days", "n_windows", "promo_quantile")})

Frozen weights : {'accuracy': 0.7, 'agreement': 0.0, 'completeness': 0.1, 'interval': 0.1, 'stability': 0.1}
Default weights: {'accuracy': 0.3, 'interval': 0.25, 'stability': 0.2, 'agreement': 0.15, 'completeness': 0.1}
Bands          : [('Accept', 80.0), ('Monitor', 60.0), ('Review', 0.0)]
Constants      : {'agree_cap': 0.5, 'cov_target': 80.0, 'cov_tol': 40.0, 'cv_cap': 1.0, 'lookback_days': 56, 'n_windows': 8, 'promo_quantile': 0.95, 'wape_cap': 50.0}


## 5. Choosing the weights on calibration, by a rule written in advance

The default weights (30/25/20/15/10) were a hypothesis. On the 8 calibration origins (96 × 8 = 768 windows) the
candidates were: the default, equal weights, **volatility-only** (stability alone, the "naive planner" index), and every
weight vector on a 0.1-step grid. Objective: **Spearman rank correlation between the score and the realised champion
WAPE of the same window**. A useful index has a strongly **negative** correlation (high trust → low error).

**Pre-stated rule** (written before results were computed): keep the default unless another candidate is at least
**0.03 more negative** *and* also better on **each half** of the calibration origins. Because completeness was 100 on
every calibration window (a constant cannot be ranked), its weight was held at the default 0.10 and only the 220 grid
points that keep it there were eligible.

In [8]:
cand = pd.DataFrame([{"candidate": c["name"], "rho (all 768)": c["spearman"], "rho 1st half": c["spearman_first_half"],
                      "rho 2nd half": c["spearman_second_half"],
                      "weights acc/int/stab/agr/comp": "/".join(f"{c['weights'][k]:.2f}" for k in TR.COMPONENTS)}
                     for c in tcal["candidates"]]).set_index("candidate")
display(cand.round(3))
d, b = cand.loc["default"], cand.loc[tcal["chosen"]]
gain = d["rho (all 768)"] - b["rho (all 768)"]
print(f"Rule check: gain {gain:.3f} >= {tcal['margin']}? {gain >= tcal['margin']} | better on 1st half? {b['rho 1st half'] < d['rho 1st half']} "
      f"| better on 2nd half? {b['rho 2nd half'] < d['rho 2nd half']}")
print("Chosen:", tcal["chosen"], "-", tcal["choice_reason"])
print(f"Default's rank among the {tcal['grid_size']} eligible grid points: {tcal['grid_rank_of_default']}")
print("Single-component rho on calibration:", tcal["single_component_spearman"])
print("Calibration bands (with hard flags):", tcal["band_distribution"], "| realised WAPE by band:", tcal["realised_wape_by_band"])

,rho (all 768),rho 1st half,rho 2nd half,weights acc/int/stab/agr/comp
candidate,,,,
default,-0.514,-0.443,-0.610,0.30/0.25/0.20/0.15/0.10
equal,-0.478,-0.402,-0.584,0.20/0.20/0.20/0.20/0.20
volatility_only,-0.449,-0.363,-0.583,0.00/0.00/1.00/0.00/0.00
grid:0.7/0.1/0.1/0.0/0.1,-0.650,-0.589,-0.731,0.70/0.10/0.10/0.00/0.10
grid:0.8/0.1/0.0/0.0/0.1,-0.646,-0.587,-0.722,0.80/0.10/0.00/0.00/0.10
grid:0.8/0.0/0.1/0.0/0.1,-0.645,-0.576,-0.741,0.80/0.00/0.10/0.00/0.10
grid:0.6/0.1/0.2/0.0/0.1,-0.643,-0.576,-0.732,0.60/0.10/0.20/0.00/0.10
grid:0.9/0.0/0.0/0.0/0.1,-0.642,-0.579,-0.732,0.90/0.00/0.00/0.00/0.10
grid:0.7/0.0/0.2/0.0/0.1,-0.640,-0.566,-0.742,0.70/0.00/0.20/0.00/0.10


Rule check: gain 0.136 >= 0.03? True | better on 1st half? True | better on 2nd half? True
Chosen: grid:0.7/0.1/0.1/0.0/0.1 - beats default by 0.136 in rho and on both calibration halves
Default's rank among the 220 eligible grid points: 62
Single-component rho on calibration: {'accuracy': -0.642, 'interval': -0.2064, 'stability': -0.4488, 'agreement': -0.19, 'completeness': None}
Calibration bands (with hard flags): {'Accept': 261, 'Monitor': 368, 'Review': 139} | realised WAPE by band: {'Accept': 9.31, 'Monitor': 15.33, 'Review': 18.62}


**Reading it:** the chosen vector puts **70 % on historical accuracy**, 10 % each on interval reliability, stability
and completeness, and **0 on model agreement**. It beat the default by well over the 0.03 margin and on both halves,
so the rule adopted it. Accuracy on its own is almost as good as the chosen mix; that is a warning sign examined in
section 7. Because 220 grid points were compared, the best calibration rho is optimistic: we should expect the
holdout rho to be a bit weaker.

## 6. Holdout validation (run once, with the frozen weights)

On the 6 holdout origins (96 × 6 = 576 windows) the frozen index is scored against what actually happened. The cell
recomputes every statistic from the saved trust scores and holdout backtests and checks it against
`holdout_results.json`.

In [9]:
realised = TR.realised_window_wape(champion_rows(bt[bt.stage == "holdout"], champs, "A"))
hs = ts[ts.origin.isin(hold_origins)].copy()
val = TR.validate(hs, realised)
saved_val = res["trust_validation"]
for k in ["n_windows", "spearman", "error_ratio_review_vs_accept", "review_share_of_abs_error", "review_share_of_windows", "volatility_only_spearman"]:
    print(f"{k:32s} recomputed {val[k]!s:>8}   saved {saved_val[k]!s:>8}")
display(pd.DataFrame(val["wape_by_band"]).set_index("band"))
print(val["verdict"])

n_windows                        recomputed      576   saved      576
spearman                         recomputed   -0.637   saved   -0.637
error_ratio_review_vs_accept     recomputed     1.96   saved     1.96
review_share_of_abs_error        recomputed     26.3   saved     26.3
review_share_of_windows          recomputed     21.4   saved     21.4
volatility_only_spearman         recomputed   -0.442   saved   -0.442


,n,wape
band,,
Accept,225,8.93
Monitor,228,17.30
Review,123,17.46


The Trust Index ranked holdout windows well: higher scores went with lower realised error (Spearman -0.64). Review windows had 2.0x the WAPE of Accept windows. The Review band held 21% of windows and 26% of absolute error. It beat the volatility-only baseline.


In [10]:
jv = hs.merge(realised, on=["series_id", "origin"])
fig = px.scatter(jv, x="score", y="wape", color="band", color_discrete_map=BAND_COLOR, opacity=0.7,
                 category_orders={"band": ["Accept", "Monitor", "Review"]},
                 hover_data={"score": ":.1f", "wape": ":.1f"},  # no series/origin ids: band membership per window stays private
                 title="Holdout: Trust score at the origin vs realised 14-day WAPE (each dot = one series-window)")
fig.update_traces(marker_size=7)
for x in (60, 80):
    fig.add_vline(x=x, line_dash="dot", line_color="#52514e")
fig.update_layout(**{**LAYOUT, "height": 440}, xaxis_title="Trust score (0-100 index)", yaxis_title="realised WAPE %",
                  yaxis_range=[0, min(100, float(jv.wape.quantile(0.995)) + 5)])
fig.show()

**Reading it:**

- **Ranking works.** The Spearman correlation between score and realised error is strongly negative on the holdout,
  close to its calibration value, and better than the volatility-only baseline.
- **Bands separate error.** Review and Monitor windows had roughly twice the WAPE of Accept windows. The Review band
  held about a fifth of the windows but a larger share of the total absolute error, so a planner who only reviews
  that band spends time where more error sits - though not overwhelmingly more.
- **Monitor and Review are hard to tell apart on realised error** (their WAPEs are almost equal). About half of the
  Review band is there because of the promo hard flag, not a low score (section 7b).
- Some high-score windows still missed badly (dots in the upper right): the index ranks *on average*; it is not a
  promise about any one window.

## 7. Concerns, checked honestly

The cells below are **hindsight diagnostics**: they use the holdout to understand the index, and none of them changed
any frozen choice.

### 7a. The weights are dominated by accuracy

In [11]:
rows = []
for c in TR.COMPONENTS:
    r, _ = TR._spearman(jv[c], jv.wape)
    rows.append({"index": f"{c} only", "holdout rho": r})
for name, w in [("default 30/25/20/15/10", C.TRUST_WEIGHTS_DEFAULT), ("frozen (chosen) weights", tw["weights"])]:
    rows.append({"index": name, "holdout rho": TR._spearman(TR.score(jv, w), jv.wape)[0]})
display(pd.DataFrame(rows).set_index("index").round(3))
print(f"Spearman between the frozen score and the accuracy component alone: {TR._spearman(jv.score, jv.accuracy)[0]:.3f}")

,holdout rho
index,
accuracy only,-0.647
interval only,0.013
stability only,-0.442
agreement only,-0.216
completeness only,NaN
default 30/25/20/15/10,-0.503
frozen (chosen) weights,-0.637


Spearman between the frozen score and the accuracy component alone: 0.971


**Reading it:** on the holdout, historical accuracy alone ranks windows as well as the full frozen score (marginally
better), and the score's rank correlation with the accuracy component alone is about 0.97. Interval reliability on its
own had essentially no ranking power on the holdout. In plain words, **the index mostly says "this
series was hard recently, so it will be hard again"**. That is useful (series difficulty is persistent) but it means
the other components add little ranking power on this data. Completeness is constant, so it has no rho. The frozen
weights still beat the default on the holdout - the choice made on calibration held up.

### 7b. The promo hard flag fires often

In [12]:
jv["promo_flag"] = jv.hard_flags.apply(lambda f: "promo_above_p95" in list(f))
jv["band_score_only"] = TR.bands_for(jv.score, None, tw["bands"]).values
pf = jv.groupby("promo_flag").apply(lambda g: pd.Series({"windows": len(g), "share %": 100 * len(g) / len(jv),
        "realised WAPE %": 100 * g.sum_abs_error.sum() / g.sum_actual.sum(), "mean score": g.score.mean()}))
display(pf.round(2))
print("Flagged windows by the band their SCORE alone would give:")
display(jv[jv.promo_flag].band_score_only.value_counts().to_frame("windows"))
by_sb = jv.groupby(["band_score_only", "promo_flag"]).apply(lambda g: 100 * g.sum_abs_error.sum() / g.sum_actual.sum()).unstack().round(2)
by_sb.columns = ["not flagged: WAPE %", "promo-flagged: WAPE %"]
forced = int((jv.promo_flag & (jv.band_score_only != "Review")).sum())
print(f"Review windows on the holdout: {int((jv.band == 'Review').sum())}, of which {forced} are there only because of the promo flag")
by_sb.reindex(["Accept", "Monitor", "Review"])

,windows,share %,realised WAPE %,mean score
promo_flag,,,,
False,509.0,88.37,13.02,74.38
True,67.0,11.63,8.71,80.08


Flagged windows by the band their SCORE alone would give:


,windows
band_score_only,
Accept,47
Monitor,18
Review,2


Review windows on the holdout: 123, of which 65 are there only because of the promo flag


,not flagged: WAPE %,promo-flagged: WAPE %
band_score_only,,
Accept,8.93,7.95
Monitor,17.30,11.23
Review,28.10,19.77


**Reading it:** the promo flag (planned promo intensity above the series' 95th percentile) fired on about 12 % of
holdout windows, because promoted-item counts trended up over 2015-2017, so the full-history 95th percentile is easy
to exceed. **On this holdout the flag pointed the wrong way:** flagged windows had *lower* realised WAPE than unflagged
ones, overall and within every score band, and almost all of them would have been Accept or Monitor on score alone.
So the flag pushed low-error forecasts into Review and diluted the Review band. It was kept as frozen (LightGBM trains
on that same history, so it was a reasonable "is this unusual?" reference), but it is the first thing to revisit,
e.g. a percentile over a recent window, or a flag only when the plan exceeds anything the model has seen.

### 7c. Completeness is a constant here

In [13]:
print("Completeness values on holdout windows:", jv.completeness.value_counts().to_dict())
print(f"Score range on holdout: {jv.score.min():.1f} - {jv.score.max():.1f} (median {jv.score.median():.1f})")
print("Holdout bands:", hs.band.value_counts().to_dict())

Completeness values on holdout windows: {100.0: 576}
Score range on holdout: 16.9 - 86.7 (median 79.5)
Holdout bands: {'Monitor': 228, 'Accept': 225, 'Review': 123}


**Reading it:** every series was recorded on every open day (notebook 01 chose the scope that way), so completeness
is 100 everywhere and its 10 % weight is a constant +10 points. It would only matter for series with recording gaps.
Also, because accuracy rarely approaches 100, scores top out in the mid-80s: Accept effectively means "near the best
historical accuracy in the portfolio".

## 8. Worked example: one series, one holdout origin, end to end

To make the machinery concrete, we trace **one real series at one holdout origin** from forecast to realised error,
recomputing every number from raw inputs and checking it against the saved scores.

Choice rule (fixed before looking at its outcome): at the **middle holdout origin** (the 3rd of 6), take the series
**without a hard flag whose Trust score is closest to the median score of that origin** - a typical, not a cherry-picked,
case.

In [14]:
T = hold_origins[2]
at_T = ts[(ts.origin == T) & (ts.hard_flags.apply(len) == 0)]
med = ts[ts.origin == T].score.median()
ex = at_T.iloc[(at_T.score - med).abs().argsort().iloc[0]]
SID, FAMILY = ex.series_id, ex.family
print(f"Origin T = {T:%Y-%m-%d} (forecasting {T + pd.Timedelta(days=1):%Y-%m-%d} .. {T + pd.Timedelta(days=14):%Y-%m-%d})")
print(f"Median score at T = {med:.2f}  ->  chosen series: {SID} (store {ex.store_nbr}, {FAMILY}), saved score {ex.score:.2f}, band {ex.band}")
print(f"Champion for {FAMILY}: {champs[FAMILY]['model']} (variant A)")

Origin T = 2017-07-11 (forecasting 2017-07-12 .. 2017-07-25)
Median score at T = 79.57  ->  chosen series: 36|DAIRY (store 36, DAIRY), saved score 79.56, band Monitor
Champion for DAIRY: lightgbm (variant A)


### Step 1 - the forecast and its interval

scale_T = mean of the last 28 days of sales up to T. For days 1-7 the family's q_lo / q_hi for bucket "1-7" apply, for
days 8-14 those of "8-14".

The computation runs on the raw panel, but everything *displayed* for this one series is an **index** (raw
competition data is not redistributed): sales-scale values ÷ the series' mean daily sales on open days
2015-06-01..2017-02-06 × 100 (100 = an average training-period day), promoted-item counts as a promo index (÷ the
series' mean count from 2015-06-01 up to T × 100). Ratios (WAPE, bias, coverage, CV) are scale-free and unchanged.

In [15]:
hist28 = panel[(panel.series_id == SID) & (panel.date > T - pd.Timedelta(days=28)) & (panel.date <= T)]
scale_T = hist28.sales.mean()
w = ci[(ci.series_id == SID) & (ci.origin == T)].sort_values("horizon").copy()
assert np.isclose(scale_T, w.scale.iloc[0])
qf = Q[Q.family == FAMILY].set_index("bucket")
w["p10 (by hand)"] = np.minimum(np.maximum(0, w.forecast + w.bucket.map(qf.q_lo) * scale_T), w.forecast)
w["p90 (by hand)"] = np.maximum(np.maximum(0, w.forecast + w.bucket.map(qf.q_hi) * scale_T), w.forecast)
assert np.allclose(w["p10 (by hand)"], w.p10) and np.allclose(w["p90 (by hand)"], w.p90)
w["inside"] = (w.actual >= w.p10) & (w.actual <= w.p90)
SB, PB = IX.series_base(panel)[SID], IX.promo_base(panel, T)[SID]   # private bases: used, never printed
wi = w.assign(**{c: IX.to_index(w[c], SB) for c in ["forecast", "p10", "p90", "actual"]},
              onpromotion=IX.to_index(w.onpromotion, PB))
wi = wi.rename(columns={"onpromotion": "promo index", "forecast": "forecast index", "p10": "P10 index",
                        "p90": "P90 index", "actual": "actual index"})
print(f"scale_T = index {IX.to_index(scale_T, SB):.1f} | {FAMILY} quantiles: 1-7 [{qf.loc['1-7','q_lo']:.3f}, {qf.loc['1-7','q_hi']:.3f}], "
      f"8-14 [{qf.loc['8-14','q_lo']:.3f}, {qf.loc['8-14','q_hi']:.3f}]  (hand-computed bands match the saved ones)")
wi[["horizon", "target_date", "promo index", "forecast index", "P10 index", "P90 index", "actual index", "inside"]].assign(target_date=lambda d: d.target_date.dt.strftime("%a %d %b")).round(1).set_index("horizon").T

scale_T = index 104.2 | DAIRY quantiles: 1-7 [-0.164, 0.172], 8-14 [-0.178, 0.149]  (hand-computed bands match the saved ones)


horizon,1,2,3,4,5,6,7,8,9,10,11,12,13,14
target_date,Wed 12 Jul,Thu 13 Jul,Fri 14 Jul,Sat 15 Jul,Sun 16 Jul,Mon 17 Jul,Tue 18 Jul,Wed 19 Jul,Thu 20 Jul,Fri 21 Jul,Sat 22 Jul,Sun 23 Jul,Mon 24 Jul,Tue 25 Jul
promo index,548.4,74.8,62.3,78.9,58.2,49.9,45.7,506.8,54.0,58.2,62.3,54.0,58.2,62.3
forecast index,105.4,79.7,76.1,105.2,134.2,95.9,105.0,103.3,80.6,77.3,99.7,127.9,91.4,100.5
P10 index,88.3,62.6,59.0,88.1,117.1,78.8,87.9,84.8,62.1,58.7,81.2,109.3,72.8,81.9
P90 index,123.3,97.6,94.0,123.1,152.1,113.8,122.9,118.8,96.1,92.8,115.2,143.4,106.9,116.0
actual index,96.4,74.0,74.4,100.0,134.0,84.5,99.4,89.1,78.2,72.9,94.5,141.0,94.3,100.3
inside,True,True,True,True,True,True,True,True,True,True,True,True,True,True


In [16]:
past = panel[(panel.series_id == SID) & (panel.date > T - pd.Timedelta(days=42)) & (panel.date <= T)]
fig = go.Figure()
X = lambda v: (IX.to_index(np.asarray(v, float), SB)).round(1)   # chart in index units, never raw units
fig.add_scatter(x=list(w.target_date) + list(w.target_date[::-1]), y=list(X(w.p90)) + list(X(w.p10)[::-1]), fill="toself",
                fillcolor="rgba(235,104,52,0.18)", line=dict(width=0), name="P10-P90 band", hoverinfo="skip")
fig.add_scatter(x=past.date, y=X(past.sales), mode="lines", name="sales index (known at T)", line=dict(color="#52514e", width=2))
fig.add_scatter(x=w.target_date, y=X(w.forecast), mode="lines+markers", name="champion forecast", line=dict(color="#eb6834", width=2), marker=dict(size=7))
fig.add_scatter(x=w.target_date, y=X(w.actual), mode="markers", name="actual (revealed later)", marker=dict(color="#2a78d6", size=9, symbol="diamond"))
fig.add_vline(x=T.timestamp() * 1000, line_dash="dash", line_color="#52514e")
fig.add_annotation(x=T, y=1.02, yref="paper", text="origin T", showarrow=False, yanchor="bottom")
fig.update_layout(**{**LAYOUT, "height": 400}, title=f"{SID}: 6 weeks of history, then the 14-day forecast with its 80% band",
                  yaxis_title="Index (training-period mean = 100)", xaxis_title=None)
fig.show()

### Step 2 - each Trust component, recomputed from raw inputs

Everything uses information available at the end of day T only.

In [17]:
K = tw["constants"]
chA = champion_rows(bt[bt.variant == "A"], champs, "A")
chA = chA[chA.series_id == SID]
done = sorted(o for o in chA.origin.unique() if o + pd.Timedelta(days=K["horizon"]) <= T)[-K["n_windows"]:]
h = evaluable(chA[chA.origin.isin(done) & (chA.target_date <= T)])
wape_hist = 100 * (h.actual - h.forecast).abs().sum() / h.actual.sum()
acc = 100 * (1 - min(wape_hist, K["wape_cap"]) / K["wape_cap"])

hiv = apply_intervals(h, panel[panel.date <= T], Q)
cov_hist = 100 * ((hiv.actual >= hiv.p10) & (hiv.actual <= hiv.p90)).mean()
inter = 100 * (1 - min(abs(cov_hist - K["cov_target"]), K["cov_tol"]) / K["cov_tol"])

l56 = panel[(panel.series_id == SID) & (panel.date > T - pd.Timedelta(days=K["lookback_days"])) & (panel.date <= T)]
ok = l56[(l56.closed == 0) & l56.sales.notna()].sales
cv56, zero56 = ok.std(ddof=0) / ok.mean(), (ok == 0).mean()
stab = 100 * (1 - min(cv56, K["cv_cap"]) / K["cv_cap"]) * (1 - zero56)

tots = bt[(bt.variant == "A") & (bt.origin == T) & (bt.series_id == SID)].groupby("model").forecast.sum()
mcv = tots.std(ddof=0) / tots.mean()
agr = 100 * (1 - min(mcv / K["agree_cap"], 1))
comp_ = 100 * len(ok) / K["lookback_days"]

roll = panel[panel.series_id == SID].sort_values("date").set_index("date").onpromotion.rolling(K["promo_window"], min_periods=K["promo_window"]).mean()
promo_p95 = roll[roll.index <= T].quantile(K["promo_quantile"])
promo_plan = w.onpromotion.mean()

print(f"History windows used (origins with o + 14 <= T): {[f'{o:%m-%d}' for o in done]}")
tots_i = IX.to_index(tots / K["horizon"], SB)   # 14-day total as index: mean daily value / base x 100
print("Five models' 14-day forecasts at T (index, mean daily / series base x 100):", {m: round(v, 1) for m, v in tots_i.items()})
steps = pd.DataFrame([
    ["accuracy", f"WAPE_hist = {wape_hist:.2f}%", f"100 × (1 − min({wape_hist:.2f}, 50)/50)", acc, ex.accuracy],
    ["interval", f"cov_hist = {cov_hist:.1f}% of {len(hiv)} days", f"100 × (1 − min(|{cov_hist:.1f} − 80|, 40)/40)", inter, ex.interval],
    ["stability", f"CV_56 = {cv56:.3f}, zero share = {zero56:.2f}", f"100 × (1 − min({cv56:.3f}, 1)) × (1 − {zero56:.2f})", stab, ex.stability],
    ["agreement", f"5 models' 14-day forecasts, mean index {tots_i.mean():.1f}", f"cv_models = {mcv:.3f} → 100 × (1 − min({mcv:.3f}/0.5, 1))", agr, ex.agreement],
    ["completeness", f"{len(ok)} of 56 days open with a sale", f"100 × {len(ok)}/56", comp_, ex.completeness],
], columns=["component", "input", "formula", "by hand", "saved"]).set_index("component")
assert np.allclose(steps["by hand"].astype(float), steps["saved"].astype(float))
assert np.isclose(promo_plan, ex.promo_plan_mean) and np.isclose(promo_p95, ex.promo_p95)
P = lambda v: IX.to_index(v, PB)
print(f"Promo flag check (promo index): planned mean {P(promo_plan):.1f} vs series p95 of 14-day mean {P(promo_p95):.1f} -> "
      f"{'FLAG' if promo_plan > promo_p95 else 'no flag'} (matches the saved plan and p95)")
steps.round(2)

History windows used (origins with o + 14 <= T): ['05-02', '05-09', '05-16', '05-23', '05-30', '06-06', '06-13', '06-27']
Five models' 14-day forecasts at T (index, mean daily / series base x 100): {'holt_winters': 101.2, 'lightgbm': 98.7, 'moving_average': 104.2, 'naive': 110.2, 'seasonal_naive': 100.5}
Promo flag check (promo index): planned mean 126.7 vs series p95 of 14-day mean 178.8 -> no flag (matches the saved plan and p95)


,input,formula,by hand,saved
component,,,,
accuracy,WAPE_hist = 11.63%,"100 × (1 − min(11.63, 50)/50)",76.73,76.73
interval,cov_hist = 73.2% of 112 days,"100 × (1 − min(|73.2 − 80|, 40)/40)",83.04,83.04
stability,"CV_56 = 0.245, zero share = 0.00","100 × (1 − min(0.245, 1)) × (1 − 0.00)",75.47,75.47
agreement,"5 models' 14-day forecasts, mean index 103.0","cv_models = 0.039 → 100 × (1 − min(0.039/0.5, 1))",92.17,92.17
completeness,56 of 56 days open with a sale,100 × 56/56,100.00,100.00


### Step 3 - score, band, and what actually happened

In [18]:
W = tw["weights"]
contrib = pd.Series({c: W[c] * steps.loc[c, "by hand"] for c in TR.COMPONENTS})
score_hand = contrib.sum()
band_hand = TR.band(score_hand, [], tw["bands"])
display(pd.DataFrame({"weight": pd.Series(W), "component": steps["by hand"].astype(float), "weight × component": contrib}).round(2))
print(f"score = {score_hand:.2f} (saved {ex.score:.2f}) -> band {band_hand} (saved {ex.band})  [an index, not a probability]")

we = evaluable(w)
win_wape = 100 * (we.actual - we.forecast).abs().sum() / we.actual.sum()
win_bias = 100 * (we.forecast - we.actual).sum() / we.actual.sum()
band_wape = {r["band"]: r["wape"] for r in val["wape_by_band"]}
print(f"Realised: 14-day WAPE {win_wape:.2f}%, bias {win_bias:+.2f}%, {int(we.inside.sum())} of {len(we)} days inside the 80% band")
print(f"For comparison, holdout WAPE of all {band_hand} windows: {band_wape[band_hand]:.2f}%; this series' own WAPE_hist was {wape_hist:.2f}%")
r = realised[(realised.series_id == SID) & (realised.origin == T)].iloc[0]
assert np.isclose(r.wape, win_wape)

,weight,component,weight × component
accuracy,0.7,76.73,53.71
agreement,0.0,92.17,0.00
completeness,0.1,100.00,10.00
interval,0.1,83.04,8.30
stability,0.1,75.47,7.55


score = 79.56 (saved 79.56) -> band Monitor (saved Monitor)  [an index, not a probability]
Realised: 14-day WAPE 6.11%, bias +3.70%, 14 of 14 days inside the 80% band
For comparison, holdout WAPE of all Monitor windows: 17.30%; this series' own WAPE_hist was 11.63%


**Reading it:** the example series is store 36 DAIRY at origin 11 July 2017. Its score of 79.56 is mostly the
accuracy component (0.7 × 76.73 = 53.71 points), i.e. its recent track record of 11.6 % WAPE; interval, stability and
the constant completeness add the rest. Model agreement was high (the five models' 14-day totals were close) but has
zero weight, so it changed nothing. The score fell just short of 80, so the band is **Monitor**, not Accept.

What happened: a 6.1 % WAPE fortnight, better than the series' own history and far better than the Monitor-band
average, with all 14 actuals inside the band (DAIRY was one of the over-covered families). A slight over-forecast
(+3.7 %) is visible in the chart. One window is one draw: a typical score can come with an unusually good or bad
fortnight, which is why the index is judged on its ranking over hundreds of windows (section 6), not on a single case.

## 9. Summary

In [19]:
vb = {r["band"]: r for r in val["wape_by_band"]}
pd.DataFrame([
    ["Interval method", "scaled split-conformal, P10/P90 per family × horizon bucket, fitted on 8 calibration origins"],
    ["Coverage: calibration in-sample / leave-one-origin-out / holdout", f"{ins.inside.mean()*100:.1f}% / {loo.inside.mean()*100:.1f}% / {ho.inside.mean()*100:.1f}% (target 80%)"],
    ["Family holdout coverage range", f"{cov.holdout.min():.1f}% ({cov.holdout.idxmin()}) - {cov.holdout.max():.1f}% ({cov.holdout.idxmax()})"],
    ["Mean band width / forecast (holdout)", f"{100 * ho.width.sum() / ho.forecast.sum():.1f}%"],
    ["Trust weights (acc/int/stab/agr/comp)", "/".join(f"{tw['weights'][c]:.1f}" for c in TR.COMPONENTS) + " (chosen on calibration by a pre-stated rule)"],
    ["Holdout Spearman(score, realised WAPE)", f"{val['spearman']:.3f} vs volatility-only {val['volatility_only_spearman']:.3f}"],
    ["Holdout WAPE: Accept / Monitor / Review", f"{vb['Accept']['wape']:.2f}% / {vb['Monitor']['wape']:.2f}% / {vb['Review']['wape']:.2f}%"],
    ["Review band: share of windows / share of absolute error", f"{val['review_share_of_windows']:.1f}% / {val['review_share_of_abs_error']:.1f}%"],
    ["Main caveats", "accuracy-dominated weights; frequent promo flag; completeness constant; 6 summer origins only"],
], columns=["what", "result"]).set_index("what")

,result
what,
Interval method,"scaled split-conformal, P10/P90 per family × h..."
Coverage: calibration in-sample / leave-one-origin-out / holdout,79.8% / 79.2% / 80.3% (target 80%)
Family holdout coverage range,72.2% (BEVERAGES) - 87.8% (GROCERY I)
Mean band width / forecast (holdout),37.3%
Trust weights (acc/int/stab/agr/comp),0.7/0.1/0.1/0.0/0.1 (chosen on calibration by ...
"Holdout Spearman(score, realised WAPE)",-0.637 vs volatility-only -0.442
Holdout WAPE: Accept / Monitor / Review,8.93% / 17.30% / 17.46%
Review band: share of windows / share of absolute error,21.4% / 26.3%
Main caveats,accuracy-dominated weights; frequent promo fla...
